In [1]:
import pandas as pd

# Se carga el CSV
df = pd.read_csv('../data/dirty_cafe_sales.csv')

# Vistazo rápido
df.head()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


In [2]:
# Dimensiones: filas x columnas
df.shape

(10000, 8)

In [3]:
# Tipos de datos y nulos por columna
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [4]:
# Estadísticas rápidas de columnas numéricas
df.describe()

,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
count,10000,9667,9862,9821,9827,7421,6735,9841
unique,10000,10,7,8,19,5,4,367
top,TXN_1961373,Juice,5,3.0,6.0,Digital Wallet,Takeaway,UNKNOWN
freq,1,1171,2013,2429,979,2291,3022,159


In [5]:
# Cuántos nulos exactos hay por columna
df.isnull().sum()

Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

In [6]:
# Busca valores tipo "ERROR" o "UNKNOWN" en columnas de texto
for col in df.select_dtypes(include=['object', 'str']).columns:
    print(col, ':', df[col].unique())

Transaction ID : <StringArray>
['TXN_1961373', 'TXN_4977031', 'TXN_4271903', 'TXN_7034554', 'TXN_3160411',
 'TXN_2602893', 'TXN_4433211', 'TXN_6699534', 'TXN_4717867', 'TXN_2064365',
 ...
 'TXN_1538510', 'TXN_3897619', 'TXN_2739140', 'TXN_4766549', 'TXN_7851634',
 'TXN_7672686', 'TXN_9659401', 'TXN_5255387', 'TXN_7695629', 'TXN_6170729']
Length: 10000, dtype: str
Item : <StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',  'UNKNOWN',
 'Sandwich',        nan,    'ERROR',    'Juice',      'Tea']
Length: 11, dtype: str
Quantity : <StringArray>
['2', '4', '5', '3', '1', 'ERROR', 'UNKNOWN', nan]
Length: 8, dtype: str
Price Per Unit : <StringArray>
['2.0', '3.0', '1.0', '5.0', '4.0', '1.5', nan, 'ERROR', 'UNKNOWN']
Length: 9, dtype: str
Total Spent : <StringArray>
[    '4.0',    '12.0',   'ERROR',    '10.0',    '20.0',     '9.0',    '16.0',
    '15.0',    '25.0',     '8.0',     '5.0',     '3.0',     '6.0',       nan,
 'UNKNOWN',     '2.0',     '1.0',     '7.5',     '4.5

## Diagnóstico de calidad de datos

El dataset mezcla tres tipos distintos de "dato faltante" en casi todas las columnas:
- `NaN`: dato genuinamente ausente
- `"UNKNOWN"`: valor no determinado al momento del registro
- `"ERROR"`: posible fallo de captura del sistema

Además, las columnas `Quantity`, `Price Per Unit`, `Total Spent` y `Transaction Date` 
están almacenadas como texto (`str`) en vez de sus tipos correctos (numérico y fecha), 
debido a esta mezcla de valores.

Hallazgo clave: `Total Spent` parece ser el resultado de `Quantity * Price Per Unit`, 
lo que sugiere una estrategia de recálculo en vez de solo imputación.

### Limpieza:
1. Convertir "UNKNOWN" y "ERROR" a NaN de forma explícita (estandarizar el "vacío")
2. Convertir columnas numéricas a su tipo correcto (float/int)
3. Convertir Transaction Date a tipo datetime
4. Recalcular Total Spent cuando sea posible usando Quantity * Price Per Unit
5. Decidir qué hacer con nulos restantes (eliminar vs imputar) según cada columna
6. Documentar cada decisión y su justificación